## Imports

In [1]:
import wandb
import logging
from tqdm import tqdm
from wandb.sdk.wandb_run import Run
import numpy as np
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
from nn_core.common import PROJECT_ROOT
import json

/leonardo_work/IscrC_SLEY/agargiul/mass/.venv/lib/python3.11/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/leonardo_work/IscrC_SLEY/agargiul/mass/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [ ]:
from mass.utils.plots import Palette

plt.rcParams.update(
    {
        "text.usetex": True,
        "font.family": "serif",
        "axes.titlesize": 24,  # Larger axes/title fonts
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 20,
        "legend.fontsize": 24,
    }
)
sns.set_context("talk")

cmap_name = "coolwarm_r"

palette = Palette(
    f"{PROJECT_ROOT}/misc/palette.json", map_path=f"{PROJECT_ROOT}/misc/palette_map.json"
)
palette

Project not installed in the current env, activate the correct env or install it with:
	pip install -e .


{'blue': '#335c67',
 'white': '#fff3b0',
 'yellow': '#e09f3e',
 'red': '#9e2a2b',
 'dark red': '#540b0e',
 'green': '#81b29a'}

## Get runs

In [30]:
api = wandb.Api()
entity, project = "gladia", "mass"  # set to your entity and project

In [31]:
def get_runs(entity, project, positive_tags, negative_tags):
    filters_pos_tags = {"$and": [{"tags": {"$eq": pos_tag}} for pos_tag in positive_tags]}
    filters_neg_tags = {}

    print(filters_pos_tags)
    filters = {**filters_pos_tags, **filters_neg_tags}
    runs = api.runs(entity + "/" + project, filters=filters)

    print(f"There are {len(runs)} runs respecting these conditions.")
    return runs

In [5]:
tags = [
    "ZeroShot"
]  

In [33]:
runs = get_runs(entity, project, positive_tags=tags, negative_tags=[])

{'$and': [{'tags': {'$eq': 'ZeroShot'}}]}
There are 9 runs respecting these conditions.


In [7]:
models = ["ViT-B-32", "ViT-B-16", "ViT-L-14"]

In [35]:
ref_run = runs[0]

In [36]:
print(set(ref_run.history().columns))

{'loss/test/Flowers102', 'normalized_acc/test/OxfordIIITPet', 'acc/test/EuroSAT', 'radar', 'loss/test/CIFAR100', 'trainer/global_step', 'loss/test/SUN397', 'loss/test/PCAM', '_step', 'normalized_acc/test/avg', 'loss/test/STL10', 'loss/test/RESISC45', 'normalized_acc/test/PCAM', 'acc/test/SUN397', 'normalized_acc/test/Flowers102', 'normalized_acc/test/GTSRB', 'normalized_acc/test/EuroSAT', 'acc/test/CIFAR100', 'acc/test/OxfordIIITPet', 'normalized_acc/test/SUN397', 'normalized_acc/test/STL10', 'acc/test/PCAM', '_runtime', 'loss/test/GTSRB', 'normalized_acc/test/FER2013', 'normalized_acc/test/CIFAR100', 'loss/test/DTD', 'normalized_acc/test/RESISC45', 'loss/test/FER2013', 'acc/test/STL10', 'acc/test/RESISC45', 'acc/test/MNIST', 'acc/test/avg', 'normalized_acc/test/SVHN', 'loss/test/Cars', 'normalized_acc/test/MNIST', 'acc/test/GTSRB', 'loss/test/OxfordIIITPet', 'normalized_acc/test/DTD', 'acc/test/DTD', 'acc/test/Cars', 'loss/test/SVHN', '_timestamp', 'epoch', 'acc/test/FER2013', 'loss/t

In [37]:
print(ref_run.config["core/tags"])

['static_merge', 'n14', 'ViT-B-32']


#### Hparams

In [38]:
benchmarks = ["n8", "n14", "n20"]
models = ["ViT-B-32", "ViT-B-16", "ViT-L-14"]

In [ ]:
avg_accs = {
    model: {benchmark: {"avg_acc": 0.0, "norm_acc": 0.0} for benchmark in benchmarks}
    for model in models
}

for run in runs:
    model = run.config["nn/encoder/model_name"]

    try:
        N = run.config["num_tasks"]
    except KeyError:
        N = run.config["ntasks"]

    benchmark = f"n{N}"

    avg_accs[model][benchmark]["avg_acc"] = run.summary["acc/test/avg"]
    avg_accs[model][benchmark]["norm_acc"] = run.summary["normalized_acc/test/avg"]

In [19]:
per_task_accs

{'ViT-B-32': {'CIFAR100': 0.7327507138252258,
  'Cars': 0.7471174597740173,
  'DTD': 0.5680271983146667,
  'EuroSAT': 0.4533582031726837,
  'FER2013': 0.5786593556404114,
  'Flowers102': 0.7395946383476257,
  'GTSRB': 0.32957184314727783,
  'MNIST': 0.4852689802646637,
  'OxfordIIITPet': 0.8985592126846313,
  'PCAM': 0.7029293179512024,
  'RESISC45': 0.6324117183685303,
  'STL10': 0.994105577468872,
  'SUN397': 0.8433693647384644,
  'SVHN': 0.32717373967170715,
  'CIFAR10': 0.9093612432479858,
  'EMNIST': 0.5036701560020447,
  'FashionMNIST': 0.6691800951957703,
  'Food101': 0.9615171551704408,
  'KMNIST': 0.1013961061835289,
  'RenderedSST2': 0.864077627658844},
 'ViT-B-16': {'Cars': 0.7544571757316589,
  'DTD': 0.5391527414321899,
  'EuroSAT': 0.5457943677902222,
  'GTSRB': 0.4384855329990387,
  'MNIST': 0.5202207565307617,
  'RESISC45': 0.6865353584289551,
  'SUN397': 0.830245316028595,
  'SVHN': 0.5336118340492249,
  'CIFAR10': 0.9187041521072388,
  'CIFAR100': 0.7430267333984375,


In [14]:
# print latex

row = '& '
for model in models:

    for benchmark in benchmarks:
        avg_acc = avg_accs[model][benchmark]["avg_acc"]
        norm_acc = avg_accs[model][benchmark]["norm_acc"]

        row += f"${avg_acc*100:.1f}_{{({norm_acc*100:.1f})}}$ & "


print(row[:-2] + '\\\\')


& $48.1_{(54.8)}$ & $56.9_{(64.5)}$ & $57.5_{(65.2)}$ & $55.3_{(60.6)}$ & $61.9_{(67.9)}$ & $62.5_{(68.3)}$ & $64.9_{(69.2)}$ & $69.1_{(73.8)}$ & $68.2_{(72.7)}$ \\
